# Experiment H2: Ensemble (Fixed)

Fix from H: expansion happens PER FOLD (no leakage).
Exactly replicates Exp G's baseline, then adds cluster ensemble on top.

In [ ]:
!git clone https://github.com/helenejabbour/ropeflow-project.git 2>/dev/null || echo 'Already cloned'
import os
BASE = 'ropeflow-project'
DATA_PROCESSED = os.path.join(BASE, 'data', 'processed')
NEW_LABELED_RAW = os.path.join(BASE, 'data', 'raw', 'new-labeled-sessions')
print('Setup done')

In [ ]:
import glob, re, json as _json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter, find_peaks
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from collections import Counter, defaultdict
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('Imports OK')

In [ ]:
CONFIG = {
    'FS': 50.0, 'CYCLE_PROMINENCE_DEGS': 100.0,
    'CYCLE_SAVGOL_WINDOW': 21, 'CYCLE_SAVGOL_POLYORDER': 3,
    'CYCLE_MIN_PEAK_DEGS': 300.0, 'CYCLE_MIN_PERIOD_S': 0.5,
    'CYCLE_MAX_PERIOD_S': 3.0, 'TARGET_LEN': 64, 'MIN_CYCLE_SAMPLES': 10,
}
KNOWN_PATTERNS = {'overhand','sneak_overhand','underhand','sneak_underhand','dragon_roll','underhand_default'}
LR_CLASSES = {'UR','UL','OR','OL','USR','USL','OSR','OSL','FB','BF'}
COARSE_MAP = {'UR':'underhand','UL':'underhand','OR':'overhand','OL':'overhand',
    'USR':'sneak_underhand','USL':'sneak_underhand','OSR':'sneak_overhand','OSL':'sneak_overhand',
    'FB':'dragon_roll','BF':'dragon_roll'}
GENERIC_TO_LR = {'U_generic':['UR','UL'],'O_generic':['OR','OL'],
    'DR_generic':['FB','BF'],'SU_generic':['USR','USL'],'SO_generic':['OSR','OSL']}
FINE_MAP = {'UR':'UR','ur':'UR','UR0':'UR','UR-CW':'UR','UL0':'UL',
    'OR':'OR','or':'OR','OR2':'OR','OR3':'OR','OR-OSL':'OR','OR/OSL?':'OR',
    'OL':'OL','ol':'OL','OL2':'OL','USR':'USR','USL':'USL','OSR':'OSR','OSL':'OSL',
    'FB':'FB','fb':'FB','FB2':'FB','BF2':'BF','bf':'BF',
    'underhand':'U_generic','overhand':'O_generic','dragon_roll':'DR_generic',
    'sneak_underhand':'SU_generic','sneak_overhand':'SO_generic',
    'CW':None,'cw':None,'CCW':None,'ccw':None,'idle':None,'Idle3':None,'Idle-or-ol?':None,'VQ5':None,'vq16':None}
_FP = [(re.compile(r'^USR$',re.I),'USR'),(re.compile(r'^USL$',re.I),'USL'),
    (re.compile(r'^OSR$',re.I),'OSR'),(re.compile(r'^OSL$',re.I),'OSL'),
    (re.compile(r'^UR',re.I),'UR'),(re.compile(r'^UL',re.I),'UL'),
    (re.compile(r'^OR',re.I),'OR'),(re.compile(r'^OL',re.I),'OL'),
    (re.compile(r'^FB',re.I),'FB'),(re.compile(r'^BF',re.I),'BF'),
    (re.compile(r'^CW$',re.I),None),(re.compile(r'^CCW$',re.I),None),
    (re.compile(r'^idle',re.I),None),(re.compile(r'^vq',re.I),None)]
def map_fine(raw):
    raw=raw.strip()
    if raw in FINE_MAP: return FINE_MAP[raw]
    if raw.lower() in FINE_MAP: return FINE_MAP[raw.lower()]
    for p,c in _FP:
        if p.match(raw): return c
    return None
def extract_signals(df):
    t=df['timestamp_ms'].values/1000.0; A=df[['ax_w','ay_w','az_w']].values
    om=df[['gx','gy','gz']].values*(np.pi/180.0); return t,A,om
def detect_cycles(t,om,fs=50.0):
    md=np.linalg.norm(om,axis=1)*(180/np.pi); n=len(md)
    if n<7: return []
    w=int(CONFIG['CYCLE_SAVGOL_WINDOW'])
    if w%2==0: w+=1
    w=max(5,min(w,n if n%2==1 else n-1)); po=max(1,min(int(CONFIG['CYCLE_SAVGOL_POLYORDER']),w-2))
    ms=savgol_filter(md,window_length=w,polyorder=po,mode='interp')
    ms=savgol_filter(ms,window_length=w,polyorder=po,mode='interp')
    pks,_=find_peaks(ms,distance=max(1,int(CONFIG['CYCLE_MIN_PERIOD_S']*fs)),
        prominence=CONFIG['CYCLE_PROMINENCE_DEGS'],height=CONFIG['CYCLE_MIN_PEAK_DEGS'])
    if len(pks)==0: return []
    cyc=[]
    for i,p in enumerate(pks):
        l=0 if i==0 else (pks[i-1]+p)//2; r=(len(t)-1) if i==len(pks)-1 else (p+pks[i+1])//2
        if r>l and (r-l)>=CONFIG['MIN_CYCLE_SAMPLES']: cyc.append((l,r))
    return cyc
def pair_cycles(t0,c0,t1,c1):
    p0,p1,u=[],[],set()
    for a in c0:
        bi,bo=-1,-1.0
        for i,b in enumerate(c1):
            if i in u: continue
            o=max(0,min(t0[a[1]],t1[b[1]])-max(t0[a[0]],t1[b[0]]))
            if o>bo: bo,bi=o,i
        if bi>=0 and bo>0: p0.append(a);p1.append(c1[bi]);u.add(bi)
    return p0,p1
def resample(s,n):
    if len(s)<2: return np.zeros(n) if s.ndim==1 else np.zeros((n,s.shape[1]))
    xo,xn=np.linspace(0,1,len(s)),np.linspace(0,1,n)
    if s.ndim==1: return np.interp(xn,xo,s)
    return np.column_stack([np.interp(xn,xo,s[:,j]) for j in range(s.shape[1])])
def build_cm(A0,A1,om0,om1,s0,e0,s1,e1):
    tl=CONFIG['TARGET_LEN']
    r0=resample(np.column_stack([A0[s0:e0],om0[s0:e0]]),tl)
    r1=resample(np.column_stack([A1[s1:e1],om1[s1:e1]]),tl)
    return np.column_stack([r0,r1]).T.astype(np.float32)
def signed_axis_features(cm):
    feats=[]
    for ch in [3,4,5,9,10,11]:
        feats.append(np.mean(cm[ch])); feats.append(np.std(cm[ch])); feats.append(np.mean(np.sign(cm[ch])))
    return np.array(feats,dtype=np.float32)
print('Functions defined')

In [ ]:
# ── Load all labeled cycles ───────────────────────────────────
all_cms=[]; all_fine=[]; all_groups=[]
def load_c(d0p,d1p,name,fine='unlabeled',windows=None):
    df0,df1=pd.read_csv(d0p),pd.read_csv(d1p)
    t0,A0,om0=extract_signals(df0); t1,A1,om1=extract_signals(df1)
    c0=detect_cycles(t0,om0,CONFIG['FS']); c1=detect_cycles(t1,om1,CONFIG['FS'])
    p0,p1=pair_cycles(t0,c0,t1,c1)
    if windows:
        fp0,fp1=[],[]
        for (s0,e0),(s1,e1) in zip(p0,p1):
            if any(ws<=(t0[s0]+t0[e0])/2<we for ws,we in windows): fp0.append((s0,e0));fp1.append((s1,e1))
        p0,p1=fp0,fp1
    g=name.split('/')[0] if '/' in name else name; cnt=0
    for (s0,e0),(s1,e1) in zip(p0,p1):
        all_cms.append(build_cm(A0,A1,om0,om1,s0,e0,s1,e1))
        all_fine.append(fine); all_groups.append(g); cnt+=1
    return cnt
for d0 in sorted(glob.glob(os.path.join(DATA_PROCESSED,'*_device0_processed.csv'))):
    d1=d0.replace('_device0_','_device1_')
    if not os.path.exists(d1): continue
    stem=os.path.basename(d0).replace('_device0_processed.csv',''); parts=stem.split('_')
    if len(parts)<3: continue
    rest='_'.join(parts[2:]); fine='unlabeled'
    for pat in sorted(KNOWN_PATTERNS,key=len,reverse=True):
        if rest.startswith(pat):
            if pat in('underhand','underhand_default'): fine='U_generic'
            elif pat=='overhand': fine='O_generic'
            elif pat=='dragon_roll': fine='DR_generic'
            elif pat=='sneak_underhand': fine='SU_generic'
            elif pat=='sneak_overhand': fine='SO_generic'
            break
    if fine=='unlabeled': continue
    load_c(d0,d1,stem,fine)
if os.path.isdir(NEW_LABELED_RAW):
    for sn in sorted(os.listdir(NEW_LABELED_RAW)):
        sd=os.path.join(NEW_LABELED_RAW,sn)
        if not os.path.isdir(sd): continue
        lp=None
        for fn in('labels_corrected.json','labels.json'):
            p=os.path.join(sd,fn)
            if os.path.isfile(p): lp=p; break
        if not lp: continue
        d0=os.path.join(DATA_PROCESSED,sn+'_device0_processed.csv')
        d1=os.path.join(DATA_PROCESSED,sn+'_device1_processed.csv')
        if not(os.path.isfile(d0) and os.path.isfile(d1)): continue
        with open(lp,encoding='utf-8') as f: data=_json.load(f)
        segs=data.get('segments',data) if isinstance(data,dict) else data
        wbl=defaultdict(list)
        for seg in segs:
            fi=map_fine(seg.get('label',''))
            if fi is None: continue
            s,e=seg.get('start'),seg.get('end')
            if s is None: continue
            if e is None: e=s+2.0
            wbl[fi].append((float(s),float(e)))
        for fi,wins in sorted(wbl.items()): load_c(d0,d1,sn+'/'+fi,fi,wins)

CMS = np.array(all_cms); y_fine = np.array(all_fine); g_all = np.array(all_groups)
X_raw = CMS.reshape(len(CMS), -1)
X_signed = np.array([signed_axis_features(cm) for cm in CMS])
lr_mask = np.array([l in LR_CLASSES for l in y_fine])
gen_mask = np.array([l in GENERIC_TO_LR for l in y_fine])

X_lr = X_raw[lr_mask]; y_lr = y_fine[lr_mask]; g_lr = g_all[lr_mask]
X_signed_lr = X_signed[lr_mask]
X_gen = X_raw[gen_mask]; y_gen = y_fine[gen_mask]; g_gen = g_all[gen_mask]
X_signed_gen = X_signed[gen_mask]

print(f'L/R: {lr_mask.sum()}, Generic: {gen_mask.sum()}')

In [ ]:
# ── Helper: expand generic labels using a trained RF ─────────
def expand_generic(X_lr_tr, y_lr_tr, X_gen_pool, y_gen_pool, g_gen_pool, X_signed_pool, thresh=0.5):
    sc = StandardScaler(); X_s = sc.fit_transform(X_lr_tr)
    rf = RandomForestClassifier(n_estimators=400, class_weight='balanced', random_state=42)
    rf.fit(X_s, y_lr_tr)
    X_gen_s = sc.transform(X_gen_pool)
    pred = rf.predict(X_gen_s)
    conf = np.max(rf.predict_proba(X_gen_s), axis=1)
    y_exp = []
    for i in range(len(X_gen_pool)):
        allowed = set(GENERIC_TO_LR.get(y_gen_pool[i], []))
        if pred[i] in allowed:
            y_exp.append(pred[i])
        else:
            probs = {c: rf.predict_proba(X_gen_s[i:i+1])[0, list(rf.classes_).index(c)]
                     for c in allowed if c in rf.classes_}
            y_exp.append(max(probs, key=probs.get) if probs else 'unknown')
    y_exp = np.array(y_exp)
    keep = (y_exp != 'unknown') & (conf >= thresh)
    return X_gen_pool[keep], y_exp[keep], g_gen_pool[keep], X_signed_pool[keep]

print('Expansion helper ready')


In [ ]:
# ── LOSO: everything happens per fold (no leakage) ───────────

unique_groups = np.unique(g_lr)
class_groups = defaultdict(set)
for l,g in zip(y_lr,g_lr): class_groups[l].add(g)
singleton = {c for c,gs in class_groups.items() if len(gs)<2}
test_groups = [g for g in unique_groups if not set(y_lr[g_lr==g]).issubset(singleton)]

strategies = ['RF only', 'Cluster only', 'Agree: use RF', 'Agree: reject', 'Agree: use cluster on disagree']
strat_data = {s: {'at10':[],'ap10':[],'at5':[],'ap5':[],'n_agree':0,'n_total':0} for s in strategies}

THRESH = 0.5

print(f'LOSO with {len(test_groups)} folds, expansion thresh={THRESH}')
for gi, g in enumerate(test_groups):
    te = g_lr == g; tr = ~te
    
    # Step 1: Expand generic labels PER FOLD (only using train L/R data)
    gen_tr_mask = g_gen != g  # generic cycles not in test group
    X_gen_tr = X_gen[gen_tr_mask]; y_gen_tr = y_gen[gen_tr_mask]; g_gen_tr = g_gen[gen_tr_mask]
    X_signed_gen_tr = X_signed_gen[gen_tr_mask]
    X_exp, y_exp, g_exp, X_signed_exp = expand_generic(X_lr[tr], y_lr[tr], X_gen_tr, y_gen_tr, g_gen_tr, X_signed_gen_tr, THRESH)
    
    # Step 2: Build training set = L/R train + expanded generic
    X_tr_all = np.vstack([X_lr[tr], X_exp])
    y_tr_all = np.concatenate([y_lr[tr], y_exp])
    X_signed_tr_all = np.vstack([X_signed_lr[tr], X_signed_exp])
    
    X_te = X_lr[te]; y_te = y_lr[te]
    X_signed_te = X_signed_lr[te]
    
    # System A: RF on raw 768D
    sc_a = StandardScaler(); X_tr_a = sc_a.fit_transform(X_tr_all); X_te_a = sc_a.transform(X_te)
    rf = RandomForestClassifier(n_estimators=400, class_weight='balanced', random_state=42)
    rf.fit(X_tr_a, y_tr_all)
    rf_pred = rf.predict(X_te_a)
    rf_conf = np.max(rf.predict_proba(X_te_a), axis=1)
    
    # System B: K-means on signed axis 18D
    sc_b = StandardScaler(); X_tr_b = sc_b.fit_transform(X_signed_tr_all); X_te_b = sc_b.transform(X_signed_te)
    km = KMeans(n_clusters=10, random_state=42, n_init=20, max_iter=500)
    km.fit(X_tr_b)
    tr_cl = km.labels_
    cl_map = {}
    for c in range(10):
        cm = tr_cl == c
        if cm.sum() == 0: cl_map[c] = 'unknown'; continue
        cl_map[c] = Counter(y_tr_all[cm]).most_common(1)[0][0]
    te_cl = km.predict(X_te_b)
    cl_pred = np.array([cl_map.get(c, 'unknown') for c in te_cl])
    
    # Evaluate strategies
    for i in range(len(y_te)):
        rp = rf_pred[i]; cp = cl_pred[i]; true = y_te[i]
        agree = (rp == cp)
        for s in strategies:
            d = strat_data[s]
            d['n_total'] += 1
            if agree: d['n_agree'] += 1
            if s == 'RF only': final = rp
            elif s == 'Cluster only': final = cp
            elif s == 'Agree: use RF': final = rp
            elif s == 'Agree: reject':
                if not agree: continue
                final = rp
            elif s == 'Agree: use cluster on disagree':
                final = rp if agree else cp
            if final == 'unknown': continue
            d['at10'].append(true); d['ap10'].append(final)
            d['at5'].append(COARSE_MAP[true]); d['ap5'].append(COARSE_MAP.get(final, final))
    
    if (gi+1) % 3 == 0: print(f'  fold {gi+1}/{len(test_groups)}')

print('Done')

In [ ]:
# ── Results ──────────────────────────────────────────────────
results = []
for s in strategies:
    d = strat_data[s]
    at10=np.array(d['at10']); ap10=np.array(d['ap10'])
    at5=np.array(d['at5']); ap5=np.array(d['ap5'])
    if len(at10) == 0:
        results.append({'name':s,'f1_10':0,'f1_5':0,'acc':0,'coverage':0,'agree_rate':0})
        continue
    labs10=sorted(set(at10)|set(ap10)); labs5=sorted(set(at5)|set(ap5))
    f1_10=f1_score(at10,ap10,average='macro',labels=labs10,zero_division=0)
    f1_5=f1_score(at5,ap5,average='macro',labels=labs5,zero_division=0)
    acc=np.mean(at10==ap10)
    cov=len(at10)/d['n_total'] if d['n_total']>0 else 0
    agr=d['n_agree']/d['n_total'] if d['n_total']>0 else 0
    results.append({'name':s,'f1_10':f1_10,'f1_5':f1_5,'acc':acc,'coverage':cov,'agree_rate':agr,
                    'cm5':confusion_matrix(at5,ap5,labels=labs5),'labs5':labs5})

In [ ]:
# ── SUMMARY ──────────────────────────────────────────────────
print('='*70)
print('EXPERIMENT H2: ENSEMBLE (FIXED) SUMMARY')
print('='*70)
print(f'Expansion: per-fold, thresh={THRESH} (no leakage)')
print(f'System A: RF 400 trees on raw 768D')
print(f'System B: K-means k=10 on signed axis 18D')
print(f'')
print(f'{"Strategy":<35s}  {"F1-10":>6s}  {"F1-5":>6s}  {"Acc":>6s}  {"Cover":>6s}  {"Agree":>6s}')
print('-'*75)
for r in sorted(results, key=lambda r: r['f1_5'], reverse=True):
    print(f'{r["name"]:<35s}  {r["f1_10"]:>6.3f}  {r["f1_5"]:>6.3f}  {r["acc"]:>6.3f}  {r["coverage"]:>5.0%}  {r["agree_rate"]:>5.0%}')
print(f'')
print(f'References:')
print(f'  Exp G (RF + expand, thresh=0.5): 5-class F1=0.823')
print(f'  E1 (RF, L/R only):               5-class F1=0.811')
print(f'  V08 (engineered features):       5-class F1=0.632')
best = max(results, key=lambda r: r['f1_5'])
print(f'')
print(f'Best: {best["name"]} (F1-5={best["f1_5"]:.3f})')
print(f'Does ensemble beat RF only? {"YES" if best["name"] != "RF only" else "NO"}')
rf_only = [r for r in results if r['name']=='RF only'][0]
print(f'RF only F1-5: {rf_only["f1_5"]:.3f} (should match Exp G: 0.823)')